# Visual Defect Lab: reproduksi di Kaggle

**Status: notebook ini belum pernah dijalankan di Kaggle.** Hasil di repositori dihitung di CPU lingkungan pengembangan. Notebook ini disediakan agar seluruh eksperimen bisa diulang di Kaggle (GPU opsional) dengan perintah yang sama.

Pengaturan Kaggle: Internet **On**, Accelerator GPU T4 opsional. Perkiraan waktu: sekitar 1 jam di CPU, lebih singkat dengan GPU.

Yang dijalankan:
1. Unduh VisA (CC BY 4.0) dan cek SHA-256.
2. Unduh bobot ImageNet resmi torchvision (ResNet18, WideResNet50) dan cek awalan SHA-256.
3. Fit dan evaluasi tiga metode: klasik OpenCV, PatchCore ResNet18, PatchCore WideResNet50.
4. Laporan bootstrap, ekspor ONNX int8, dan pemeriksaan independen.
5. Bandingkan dengan `evaluation/results.json` yang ada di repositori.

In [ ]:
!git clone --depth 1 https://github.com/muhshandy-dev/visual-defect-lab.git
%cd visual-defect-lab
!pip install -q -e ".[model,dev]"
import json, shutil
shutil.copy("evaluation/results.json", "/kaggle/working/results_repo.json")

In [ ]:
import os, torch
# Hasil di repo dibuat dengan cpu. GPU mempercepat tetapi angka bisa berbeda sangat sedikit.
os.environ["DEFECT_LAB_DEVICE"] = "cuda" if torch.cuda.is_available() else "cpu"
print(os.environ["DEFECT_LAB_DEVICE"])

In [ ]:
!mkdir -p data/weights
!wget -q -P data/weights https://download.pytorch.org/models/resnet18-f37072fd.pth
!wget -q -P data/weights https://download.pytorch.org/models/wide_resnet50_2-95faca4d.pth
!sha256sum data/weights/*.pth

In [ ]:
!defect-lab data

In [ ]:
!defect-lab evaluate --method classical
!defect-lab evaluate --method resnet18
!defect-lab evaluate --method wrn50

In [ ]:
!defect-lab report
!defect-lab export-onnx
!defect-lab export-web
!python tools/verify_published.py

In [ ]:
new = json.load(open("evaluation/results.json"))
old = json.load(open("/kaggle/working/results_repo.json"))
for m in new["methods"]:
    if m not in old["methods"]:
        continue
    for k, v in new["methods"][m]["macro"].items():
        o = old["methods"][m]["macro"][k]
        print(f"{m:9s} {k:12s} repo {o:.4f}  kaggle {v:.4f}  selisih {v - o:+.4f}")